# 04 · 자연어 기반 취합 → 위키 Markdown 생성

**파이프라인 4/4** — 자연어 질의 한 줄로 두 MCP에서 관련 데이터만 취합하고(에이전트),
LLM으로 위키용 Markdown 초안을 생성합니다. 검토·수정 후 저장/커밋합니다.

전체 흐름(**extract → generate → save/commit**)이 아래 셋업 셀들에 인라인되어 있어
이 노트북만으로 파이프라인 전체를 이해·실행할 수 있습니다. 출력은 노트북 전용
`notebook/wiki/`에 저장됩니다(웹앱의 `app/wiki`와 분리).

## 셋업 (설정 + 인증 + 에이전트 + 위키 생성)

In [ ]:
# ============================================================
# 셋업 — 이 셀 하나로 설정·인증 준비 (pipeline/*.py 없이 독립 실행)
# ============================================================
import os, json, asyncio
from pathlib import Path
from contextlib import AsyncExitStack

import msal
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client, create_mcp_http_client

# 1) 프로젝트 루트를 찾아 .env 로드 (.env.example/requirements.txt가 있는 폴더)
def find_project_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / '.env.example').is_file() or (base / 'requirements.txt').is_file():
            return base
    return Path.cwd()

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / '.env')
TOKEN_CACHE = PROJECT_ROOT / '.token_cache.json'   # 로그인 1회 → 이후 노트북/웹앱이 재사용

# 2) 설정값 (.env에서 읽음)
TENANT_ID     = os.environ.get('TENANT_ID') or '00000000-0000-0000-0000-000000000000'
CLIENT_ID     = os.environ.get('CLIENT_ID', '')
CLIENT_SECRET = os.environ.get('CLIENT_SECRET', '')
AUTH_MODE     = os.environ.get('AUTH_MODE', 'device_code')   # device_code | client_credentials
AUTHORITY     = f'https://login.microsoftonline.com/{TENANT_ID}'

def _mcp_url(env_var, server_id):
    return (os.environ.get(env_var) or '').strip() or \
        f'https://agent365.svc.cloud.microsoft/agents/tenants/{TENANT_ID}/servers/{server_id}'

# 두 MCP 소스: Mail / Teams. 서버마다 delegated 스코프가 달라 토큰도 소스별로 1개씩.
SOURCES = {
    'mail':  {'label': 'Mail',  'url': _mcp_url('MAIL_MCP_SERVER_URL',  'mcp_MailTools')},
    'teams': {'label': 'Teams', 'url': _mcp_url('TEAMS_MCP_SERVER_URL', 'mcp_TeamsServer')},
}
for _s in SOURCES.values():
    _s['scopes'] = [f"{_s['url']}/.default"]

# 3) MSAL 인증 — 소스별 access token. device_code는 최초 1회만 로그인, 이후 캐시 재사용.
_cache = msal.SerializableTokenCache()
if TOKEN_CACHE.exists():
    _cache.deserialize(TOKEN_CACHE.read_text(encoding='utf-8'))

def _save_cache():
    if _cache.has_state_changed:
        TOKEN_CACHE.write_text(_cache.serialize(), encoding='utf-8')

def ensure_token(source_key: str) -> str:
    """소스 1개의 access token 확보 (없으면 device-code 로그인)."""
    assert CLIENT_ID, 'CLIENT_ID가 .env에 없습니다 (.env.example 참고).'
    scopes = SOURCES[source_key]['scopes']
    if AUTH_MODE == 'client_credentials':           # 앱 전용 토큰
        app = msal.ConfidentialClientApplication(
            CLIENT_ID, authority=AUTHORITY, client_credential=CLIENT_SECRET, token_cache=_cache)
        res = app.acquire_token_for_client(scopes=scopes)
    else:                                           # 위임(사용자) 로그인
        app = msal.PublicClientApplication(CLIENT_ID, authority=AUTHORITY, token_cache=_cache)
        accounts = app.get_accounts()
        res = app.acquire_token_silent(scopes, account=accounts[0]) if accounts else None
        if not (res and 'access_token' in res):     # 캐시에 없으면 device-code 로그인
            flow = app.initiate_device_flow(scopes=scopes)
            print(flow['message'])                   # https://microsoft.com/devicelogin + 코드 안내
            res = app.acquire_token_by_device_flow(flow)
    _save_cache()
    if not res or 'access_token' not in res:
        raise RuntimeError((res or {}).get('error_description', 'access token 획득 실패'))
    return res['access_token']

print('PROJECT_ROOT:', PROJECT_ROOT)
print('TENANT_ID   :', TENANT_ID, '| CLIENT_ID set:', bool(CLIENT_ID), '| AUTH_MODE:', AUTH_MODE)
for _k, _s in SOURCES.items():
    print(f"  [{_k}] {_s['label']} -> {_s['url']}")

In [ ]:
# --- LLM + 에이전트: 여러 MCP 소스에 연결해 LLM이 도구를 호출하는 루프 ---
def tool_schema(tool):
    sc = getattr(tool, 'inputSchema', None) or {'type': 'object', 'properties': {}}
    return sc.model_dump() if hasattr(sc, 'model_dump') else sc

def content_to_text(result) -> str:
    parts = []
    for b in (getattr(result, 'content', None) or []):
        if getattr(b, 'type', None) == 'text':
            t = getattr(b, 'text', '')
            try:                                    # 서버가 \uXXXX 이스케이프 JSON을 주면 한글로 풀어서 넣는다
                t = json.dumps(json.loads(t), ensure_ascii=False, indent=2)
            except Exception:
                pass
            parts.append(t)
        else:
            data = b.model_dump() if hasattr(b, 'model_dump') else b
            parts.append('```json\n' + json.dumps(data, ensure_ascii=False, indent=2, default=str) + '\n```')
    text = '\n'.join(parts)
    return f'ERROR: {text}' if getattr(result, 'isError', False) else text

def make_openai():
    """Azure OpenAI(Entra 키리스=az login/Managed Identity 우선, 없으면 API 키) 또는 OpenAI 클라이언트를 만든다."""
    ep = (os.environ.get('AZURE_OPENAI_ENDPOINT') or '').strip()
    dep = (os.environ.get('AZURE_OPENAI_DEPLOYMENT') or '').strip()
    if ep and dep:
        from openai import AzureOpenAI
        common = dict(azure_endpoint=ep, azure_deployment=dep,
                      api_version=os.environ.get('AZURE_OPENAI_API_VERSION', '2024-10-21'))
        key = (os.environ.get('AZURE_OPENAI_API_KEY') or '').strip()
        if key:                                        # API 키가 있으면 키 기반
            return {'client': AzureOpenAI(api_key=key, **common), 'model': dep}
        # 키가 비었으면 Entra 키리스(az login / Managed Identity)로 인증한다.
        # 주의: .env의 빈 AZURE_OPENAI_API_KEY("")가 환경에 남아 있으면 openai SDK의
        # 자격증명 검사를 오염시켜 키리스가 막히므로, 빈 값은 환경에서 제거한다.
        os.environ.pop('AZURE_OPENAI_API_KEY', None)
        from azure.identity import DefaultAzureCredential, get_bearer_token_provider
        tp = get_bearer_token_provider(DefaultAzureCredential(), 'https://cognitiveservices.azure.com/.default')
        return {'client': AzureOpenAI(azure_ad_token_provider=tp, **common), 'model': dep}
    okey = (os.environ.get('OPENAI_API_KEY') or '').strip()
    if okey:
        from openai import OpenAI
        return {'client': OpenAI(api_key=okey), 'model': os.environ.get('OPENAI_MODEL', 'gpt-4o-mini')}
    raise RuntimeError('LLM 미설정 — .env에 AZURE_OPENAI_* 또는 OPENAI_API_KEY를 넣으세요.')

async def run_agent(tokens_by_source: dict, user_message: str, max_iters: int = 8) -> dict:
    oa = make_openai()
    trace, source_errors = [], {}
    async with AsyncExitStack() as stack:
        sessions, tools, registry = {}, [], {}
        # 1) 각 소스에 연결 + 도구 목록을 OpenAI 함수 스펙으로 변환
        for key, token in tokens_by_source.items():
            try:
                http_client = await stack.enter_async_context(
                    create_mcp_http_client(headers={'Authorization': f'Bearer {token}'}))
                r, w, _ = await stack.enter_async_context(
                    streamable_http_client(SOURCES[key]['url'], http_client=http_client))
                s = await stack.enter_async_context(ClientSession(r, w))
                await s.initialize()
                sessions[key] = s
                for t in (await s.list_tools()).tools or []:
                    fname = f'{key}__{t.name}'[:64]          # 소스 네임스페이스로 이름 고유화
                    registry[fname] = (key, t.name)
                    tools.append({'type': 'function', 'function': {
                        'name': fname,
                        'description': f"[{SOURCES[key]['label']}] {t.description or t.name}",
                        'parameters': tool_schema(t)}})
            except Exception as exc:
                source_errors[key] = str(exc)
        if not tools:
            return {'answer': '사용할 수 있는 도구가 없습니다. 연결 상태를 확인하세요.',
                    'trace': trace, 'source_errors': source_errors}
        # 2) LLM ↔ 도구 호출 루프
        messages = [
            {'role': 'system', 'content':
                'You are connected to Microsoft "Work IQ" MCP servers. Tool names are prefixed by '
                'source ("mail__"/"teams__"). Prefer calling a tool over guessing; resolve names to '
                'real IDs first and never invent IDs. Answer in the user\'s language, concisely.'},
            {'role': 'user', 'content': user_message}]
        for _ in range(max_iters):
            comp = await asyncio.to_thread(
                oa['client'].chat.completions.create,
                model=oa['model'], messages=messages, tools=tools, tool_choice='auto', temperature=0)
            msg = comp.choices[0].message
            m = {'role': 'assistant', 'content': msg.content or ''}
            if msg.tool_calls:
                m['tool_calls'] = [{'id': c.id, 'type': 'function', 'function': {
                    'name': c.function.name, 'arguments': c.function.arguments}} for c in msg.tool_calls]
            messages.append(m)
            if not msg.tool_calls:                      # 도구 호출이 없으면 최종 답변
                return {'answer': msg.content or '(no content)', 'trace': trace, 'source_errors': source_errors}
            for c in msg.tool_calls:                     # 모델이 고른 도구를 실제 세션으로 라우팅
                try:
                    args = json.loads(c.function.arguments or '{}')
                except Exception:
                    args = {}
                key, orig = registry.get(c.function.name, (None, None))
                if key is None:
                    out = f'ERROR: unknown tool {c.function.name}'
                else:
                    try:
                        out = content_to_text(await sessions[key].call_tool(orig, arguments=args))
                    except Exception as exc:
                        out = f'ERROR calling {orig}: {exc}'
                trace.append({'tool': orig, 'source': key, 'args': args, 'result': out})
                messages.append({'role': 'tool', 'tool_call_id': c.id, 'content': out[:8000]})
        return {'answer': '도구 호출 한도(max_iters) 초과.', 'trace': trace, 'source_errors': source_errors}

In [ ]:
# --- 위키 문서 생성(LLM) + 저장/커밋 헬퍼 ---
import re, subprocess
from datetime import date, datetime, timezone, timedelta

WIKI_DIR = PROJECT_ROOT / 'notebook' / 'wiki'      # 노트북 전용 출력 폴더 (app/과 무관)

_WRITER_PROMPT = (
    '너는 사내 엔지니어링 지식 위키를 관리하는 시니어 테크니컬 라이터다. 수집된 원본(팀즈/메일)을 '
    '하나의 잘 구조화된 Markdown 위키 문서로 정리해라.\n'
    '- 첫 줄은 H1 제목 한 줄: `# <간결한 제목>`\n'
    '- 원본 언어(한국어면 한국어)로 작성하고 H2/H3로 개요/방법/예시/주의사항 등 구성\n'
    '- 명령어·코드·설정은 코드블록에 원문 그대로 보존, 잡담·인사·불필요한 PII는 제거\n'
    '- 사실을 지어내지 말 것. 마지막에 `## 출처` 섹션(소스/작성자/날짜/ID)을 둔다\n'
    '- 문서 Markdown만 출력 (앞뒤 설명이나 전체 코드펜스 없이)')

def default_date_range(days=1):
    """최근 days일 범위 (start, end) ISO 날짜. days=1 => 어제..오늘."""
    end = date.today(); start = end - timedelta(days=max(days, 0))
    return start.isoformat(), end.isoformat()

def build_extraction_prompt(query, start, end, source_keys):
    labels = ', '.join(SOURCES[k]['label'] for k in source_keys)
    return (f'지식 위키용 자료를 수집한다. 아래 주제와 관련되고 재사용 가능한 기술 지식/노하우'
            f'(how-to, 결정사항, 트러블슈팅, 팁, 설정, 교훈)를 담은 메시지/메일만 모아라.\n\n'
            f'주제:\n{query}\n\n기간(포함): {start} .. {end}\n검색 소스: {labels}\n\n'
            '1) search/list 도구로 기간 내 후보를 찾고, 2) 주제와 무관한 잡담·일정은 버리고, '
            '3) 남긴 항목마다 소스/작성자/시각/제목/실제 ID/유용한 내용 요약(명령·코드는 원문 인용)을 정리하고, '
            '4) 관련 자료가 없으면 없다고 명시. 대화체가 아니라 위키 원자료로 정확하게 소스별로 묶어 반환.')

async def extract_knowledge(tokens, query, start, end):
    """추출용 프롬프트로 에이전트를 돌려 원자료를 모은다."""
    prompt = build_extraction_prompt(query, start, end, list(tokens.keys()))
    return await run_agent(tokens, prompt)

async def to_markdown(query, start, end, source_keys, material):
    """수집한 원자료(material)를 위키 Markdown 문서로 생성 (저장/커밋은 안 함)."""
    oa = make_openai()
    user = (f'추출 질의: {query}\n범위: {start} .. {end}\n'
            f"출처: {', '.join(SOURCES[k]['label'] for k in source_keys)}\n\n"
            f'=== 수집된 원본 자료 ===\n{material}')
    comp = await asyncio.to_thread(
        oa['client'].chat.completions.create, model=oa['model'],
        messages=[{'role': 'system', 'content': _WRITER_PROMPT}, {'role': 'user', 'content': user}],
        temperature=0.2)
    body = comp.choices[0].message.content or ''
    title = next((l[2:].strip() for l in body.splitlines() if l.startswith('# ')), query or '제목 없음')
    slug = re.sub(r'-{2,}', '-', re.sub(r'[^\w가-힣]+', '-', title.lower(), flags=re.UNICODE)).strip('-') or 'untitled'
    src_list = '[' + ', '.join('"' + SOURCES[k]['label'] + '"' for k in source_keys) + ']'
    fm = ('---\n'
          f'title: "{title}"\n'
          f'generated: "{datetime.now(timezone.utc).isoformat(timespec="seconds")}"\n'
          f'sources: {src_list}\n'
          f'date_range: "{start}..{end}"\n'
          f'query: "{query}"\n'
          'generator: "llmwiki-notebook"\n'
          '---')
    return {'title': title, 'slug': slug, 'markdown': f'{fm}\n\n{body.strip()}\n'}

def save_doc(markdown, slug):
    WIKI_DIR.mkdir(parents=True, exist_ok=True)
    path = WIKI_DIR / f'{date.today().isoformat()}-{slug}.md'
    path.write_text(markdown, encoding='utf-8')
    return path

def save_and_commit(markdown, slug, message=None):
    """문서를 저장하고 그 파일만 git 커밋 (pathspec 제한)."""
    path = save_doc(markdown, slug)
    def git(*a):
        return subprocess.run(['git', *a], cwd=str(PROJECT_ROOT), capture_output=True, text=True)
    git('add', '--', str(path))
    c = git('commit', '-m', message or f'docs(wiki): add {slug}', '--', str(path))
    return {'path': str(path), 'commit': (c.stdout + c.stderr).strip() or '(변경 없음/실패)'}

def list_docs():
    if not WIKI_DIR.exists():
        return []
    return [{'filename': p.name, 'path': str(p)} for p in sorted(WIKI_DIR.glob('*.md'), reverse=True)]

## 1. 토큰 확보 + 추출 범위 지정

In [ ]:
tokens = {}
for key in SOURCES:
    try:
        tokens[key] = ensure_token(key)
    except Exception as exc:
        print(f'{key} 토큰 실패: {exc}')
make_openai()   # LLM 설정 확인

QUERY = 'Postgres 튜닝, Kubernetes 트러블슈팅, CI 캐시 등 재사용 가능한 기술 노하우'
START, END = default_date_range(days=7)   # 최근 7일 (넓히려면 days 증가)
print('질의:', QUERY)
print('범위:', START, '..', END)
print('소스:', list(tokens.keys()))

## 2. 추출 (extract) — 에이전트로 관련 자료 취합
선택한 소스에서 주제·기간에 맞는 항목만 모읍니다. LLM/토큰이 필요합니다.

In [ ]:
extract_result = await extract_knowledge(tokens, QUERY, START, END)
if extract_result['source_errors']:
    print('소스 경고:', json.dumps(extract_result['source_errors'], ensure_ascii=False, indent=2))
material = extract_result['answer']
print(material[:2000])

## 3. 생성 (generate) — 위키 Markdown 초안 만들기 (저장 안 함)

In [ ]:
doc = await to_markdown(QUERY, START, END, list(tokens.keys()), material)
print('제목:', doc['title'])
print('slug:', doc['slug'])
markdown_text = doc['markdown']

## 4. 초안 미리보기

In [ ]:
from IPython.display import Markdown, display
import re
# 저장본에는 YAML front matter(---...---)가 들어가지만, 미리보기에서는 지저분하니 본문만 렌더링한다
_preview = re.sub(r'\A\ufeff?---\r?\n.*?\r?\n---\r?\n+', '', markdown_text, count=1, flags=re.DOTALL)
display(Markdown(_preview))

## 5. 검토·수정
아래 `markdown_text` 문자열을 직접 편집하세요. (웹앱에서는 편집 UI로 수행)

In [ ]:
# 예: markdown_text = markdown_text.replace('오타', '수정')
print(markdown_text[:800])

## 6. 저장 + 커밋 → `notebook/wiki/`
검토가 끝나면 저장/커밋합니다. 커밋은 **해당 문서 파일만** 포함합니다(pathspec 제한).
저장만 하려면 `save_doc(...)`을 쓰세요.

In [ ]:
DO_COMMIT = False   # 실제로 커밋하려면 True

if DO_COMMIT:
    out = save_and_commit(markdown_text, doc['slug'])
    print('저장:', out['path'])
    print('커밋:', out['commit'])
else:
    path = save_doc(markdown_text, doc['slug'])
    print('저장만 완료(커밋 안 함):', path)

## 7. 저장된 위키 문서 목록

In [ ]:
for d in list_docs():
    print(f"- {d['filename']}")

🎉 완료! 생성된 Markdown이 `notebook/wiki/`에 저장됩니다. 동일한 흐름을 웹 UI로 쓰려면
프로젝트 루트에서 `uvicorn app.main:app --reload` 실행 후 브라우저로 접속하세요.